# Notebook 1 — Agregación de la tabla clínica a nivel de paciente

**Objetivo.** Convertir la tabla clínica original (una fila por línea de tratamiento) en una tabla con **una fila por paciente**, lista para el modelado de ypN (afectación ganglionar axilar tras terapia neoadyuvante) en cáncer de mama localmente avanzado. Se define la cohorte final: pacientes con datos clínicos e imagen PET-CT y sin metástasis a distancia (M0).

**Entradas**
- `data/clinical_data.csv`: tabla clínica original (separador `;`).
- `data/image_metadata.csv`: metadatos de imagen, para identificar quién tiene PET-CT.

**Salidas**
- `data/tabla_clinica_agregada.csv`: una fila por paciente (si `EXCLUIR_M1 = False`, el archivo lleva el sufijo `_conM1`).

**Aislamiento / no data leakage.** Este notebook solo reestructura y filtra con criterios clínicos y de disponibilidad de datos. No aprende nada de los datos (ni imputación, ni escalado, ni selección de variables), así que no puede haber fuga de información hacia la evaluación. La imputación y la selección de variables se hacen más adelante, dentro de cada fold de la validación cruzada.

In [1]:
## Sección 1. Carga de datos y descripción de la tabla original

In [2]:
import pandas as pd

In [3]:
ruta_clinica = "data/clinical_data.csv"

 El CSV original usa ';' como separador (formato habitual de Excel en español).
 Cada fila es una línea de tratamiento, no un paciente: de ahí las 389 filas para 100 pacientes.

In [4]:
df = pd.read_csv(ruta_clinica, sep=';')

In [5]:
print("Total filas:", len(df))

Total filas: 389


In [6]:
print("Pacientes únicos:", df['Patient ID'].nunique())

Pacientes únicos: 100


In [7]:
print("\nDistribución de nº de filas por paciente:")
print(df.groupby('Patient ID').size().value_counts().sort_index())


Distribución de nº de filas por paciente:
1     1
2     8
3    32
4    24
5    30
6     5
Name: count, dtype: int64


In [8]:
## Sección 2. Comprobación de consistencia entre filas del mismo paciente

In [9]:
print("\nConsistencia de ypN por paciente:")
inconsistencias = df.groupby('Patient ID')['ypN'].nunique()


Consistencia de ypN por paciente:


In [10]:
print("Pacientes con valores de ypN distintos entre filas:", (inconsistencias > 1).sum())

Pacientes con valores de ypN distintos entre filas: 0


 --- Comprobación EXTENDIDA: ¿hay contradicciones (2+ valores no vacíos distintos)
     en CUALQUIER variable "por paciente", no solo ypN? ---

In [11]:
print("\n[VERIFICACIÓN DE CONTRADICCIONES en todas las variables 'por paciente']")


[VERIFICACIÓN DE CONTRADICCIONES en todas las variables 'por paciente']


In [12]:
variables_por_paciente = [
    'Patient sex', 'Age at diagnosis', 'Cancer type', 'Tumor histology',
    'HER2 (presence)', 'ER (presence)', 'PR (presence)', 'Ki-67',
    'Size of primary lesion', 'T', 'N', 'M', 'SUV max (at diagnosis)',
    'Systemic therapy breast response (RECIST)', 'Systemic therapy axillar response (RECIST)',
    'ypT', 'ypN', 'Delta SUV max', 'Delta TLG', 'Delta MTV',
    'Patient death', 'Recurrence'
]

In [13]:
hay_contradicciones = False
for var in variables_por_paciente:
    if var not in df.columns:
        continue
    n_valores_distintos_no_nulos = df.groupby('Patient ID')[var].apply(lambda s: s.dropna().nunique())
    pacientes_con_contradiccion = (n_valores_distintos_no_nulos > 1).sum()
    if pacientes_con_contradiccion > 0:
        hay_contradicciones = True
        print(f"  [AVISO] '{var}': {pacientes_con_contradiccion} paciente(s) con valores CONTRADICTORIOS (no solo NA/valor, sino dos valores reales distintos)")

In [14]:

if not hay_contradicciones:
    print("  Ninguna variable presenta contradicciones — 'primer_valor_valido' es seguro de usar en todas ellas.")


  Ninguna variable presenta contradicciones — 'primer_valor_valido' es seguro de usar en todas ellas.


In [15]:
print("\nValores únicos de 'Population':", df['Population'].unique())
print("Valores únicos de 'Site':", df['Site'].unique())


Valores únicos de 'Population': ['patient with cancer' nan]
Valores únicos de 'Site': ['tMMMs8156h' nan]


**Verificación del patrón fármacos simultáneos vs. fases secuenciales**

In [16]:
print("\nEjemplo de un paciente con varias líneas de tratamiento:")
paciente_multi = df['Patient ID'].value_counts()
paciente_multi = paciente_multi[paciente_multi > 1].index
if len(paciente_multi) > 0:
    ejemplo = df[df['Patient ID'] == paciente_multi[0]]
    #print(ejemplo[['Patient ID', 'Systemic therapy drug', 'Systemic therapy cycles', 'Systemic therapy start date']].to_string())


Ejemplo de un paciente con varias líneas de tratamiento:


**Sección 3.** **Funciones de agregación (valor válido y tratamiento por familia de fármacos).**

Función de agregación robusta ante NA: coge el primer valor NO VACÍO ---
 Necesaria porque, al ser filas por línea de tratamiento, algunas variables
"por paciente" (ypN, ypT, sexo, etc.) pueden estar rellenas solo en ALGUNAS
 de las filas de ese paciente, no en todas. Usar 'first' a secas cogería
 la primera fila sin más, perdiendo el dato real si esa fila concreta
 tenía NA en esa columna.

In [17]:
def primer_valor_valido(serie):
    valores_no_nulos = serie.dropna()
    return valores_no_nulos.iloc[0] if len(valores_no_nulos) > 0 else pd.NA

Es seguro porque la comprobación anterior mostró que ninguna variable tiene valores contradictorios entre filas.

 *Clasificación de fármacos por categoría terapéutica (18 fármacos -> 10 familias)*
 
Evita mezclar ciclos/tratamientos de naturaleza distinta (ej. taxano + antraciclina) en un único número sin sentido clínico. Cada categoría se trata de forma independiente: variable binaria (recibió sí/no) + ciclos SOLO dentro de esa
 misma categoría (nunca sumados entre categorías distintas). Agrupar en familias evita variables con pocos casos (un fármaco concreto rara vez supera 10 pacientes) y da sentido clínico a los ciclos.

In [18]:
CATEGORIAS_FARMACOS = {
    'taxano': ['Paclitaxel', '(nab)Paclitaxel', 'Docetaxel'],
    'antraciclina': ['Epirubicin', 'Anthracycline'],
    'alquilante': ['Cyclophosphamide'], 
    'platino': ['Carboplatin'],
    'anti_HER2': ['Trastuzumab', 'Pertuzumab', 'Emtansine'],
    'hormonoterapia': ['Letrozole', 'Fulvestrant'],
    'inhibidor_CDK46': ['Ribociclib', 'Palbociclib'],
    'inmunoterapia': ['Atezolizumab', 'Pembrolizumab'],
    'ipatasertib': ['Ipatasertib'],
    'soporte_GCSF': ['GCSF'],  # no antitumoral, factor de soporte hematológico
}

In [19]:
set(df['Systemic therapy drug'].dropna()) - {f for l in CATEGORIAS_FARMACOS.values() for f in l}

set()

In [20]:
sin_clasificar = set(df['Systemic therapy drug'].dropna()) - {f for l in CATEGORIAS_FARMACOS.values() for f in l}
print(sin_clasificar)
print(df[df['Systemic therapy drug'].isin(sin_clasificar)]['Patient ID'].nunique(), "pacientes lo recibieron")

set()
0 pacientes lo recibieron


In [21]:
def procesar_tratamiento_por_categoria(grupo):
    """
    Para un paciente (grupo de filas), calcula:
      - recibio_<categoria>: booleano, si recibió algún fármaco de esa familia
      - ciclos_<categoria>: suma de ciclos SOLO dentro de esa familia,
        agrupando primero por fecha de inicio (fármacos simultáneos no se
        duplican) y sumando solo entre fechas distintas (fases secuenciales)
    """
    resultado = {}
    farmacos_paciente = grupo['Systemic therapy drug'].dropna().astype(str).tolist()

    for categoria, lista_farmacos in CATEGORIAS_FARMACOS.items():
        mascara_categoria = grupo['Systemic therapy drug'].isin(lista_farmacos)
        recibio_categoria = mascara_categoria.any()
        resultado[f'{categoria}'] = bool(recibio_categoria)

        if recibio_categoria:
            subgrupo = grupo[mascara_categoria]
            ciclos_por_fecha = subgrupo.groupby('Systemic therapy start date')['Systemic therapy cycles'].first()
            resultado[f'ciclos_{categoria}'] = ciclos_por_fecha.sum()
        else:
            resultado[f'ciclos_{categoria}'] = 0

    return pd.Series(resultado)


resultado_tratamiento = df.groupby('Patient ID').apply(
    procesar_tratamiento_por_categoria,
    include_groups=False
).reset_index()


*Agregación del resto de variables (constantes por paciente, robusta ante NA)*

In [22]:
agregaciones = {
    'Patient sex': primer_valor_valido,
    'Age at diagnosis': primer_valor_valido,
    'Cancer type': primer_valor_valido,
    'Tumor histology': primer_valor_valido,
    'HER2 (presence)': primer_valor_valido,
    'ER (presence)': primer_valor_valido,
    'PR (presence)': primer_valor_valido,
    'Ki-67': primer_valor_valido,
    'Size of primary lesion': primer_valor_valido,
    'T': primer_valor_valido, 'N': primer_valor_valido, 'M': primer_valor_valido,
    'SUV max (at diagnosis)': primer_valor_valido,
    'Systemic therapy breast response (RECIST)': primer_valor_valido,
    'Systemic therapy axillar response (RECIST)': primer_valor_valido,
    'ypT': primer_valor_valido,
    'ypN': primer_valor_valido,
    'Delta SUV max': primer_valor_valido,
    'Delta TLG': primer_valor_valido,
    'Delta MTV': primer_valor_valido,
    'Patient death': primer_valor_valido,
    'Recurrence': primer_valor_valido,
}

In [23]:
## Sección 4. Agregación a una fila por paciente

In [24]:
df_paciente = df.groupby('Patient ID').agg(agregaciones).reset_index()

In [25]:
# --- Unir con la info de tratamiento ya calculada correctamente ---
df_paciente = df_paciente.merge(resultado_tratamiento, on='Patient ID')

**Sección 5. Cohorte final: pacientes con datos clínicos e imagen**

*Filtrar a la cohorte real con imagen disponible (intersección clínico+imagen = 92)*
 
Verificado exhaustivamente: la discrepancia entre 100 (clínico) y 96 (imagen) no es un
problema de formato de ID, es una diferencia real de disponibilidad de datos.

In [26]:
RUTA_METADATA_IMAGEN = "data/image_metadata.csv"
df_metadata_img = pd.read_csv(RUTA_METADATA_IMAGEN)
pacientes_con_imagen = set(df_metadata_img['PatientID'].astype(str).str.strip())
pacientes_clinicos_agregados = set(df_paciente['Patient ID'].astype(str).str.strip())

Verificación EXPLÍCITA de la intersección, para dejar claro qué se excluye y por qué:

In [27]:
solo_clinico_excluidos = pacientes_clinicos_agregados - pacientes_con_imagen
solo_imagen_excluidos = pacientes_con_imagen - pacientes_clinicos_agregados
interseccion = pacientes_clinicos_agregados & pacientes_con_imagen

In [28]:
print(f"\n[FILTRO COHORTE FINAL — verificación explícita]")
print(f"Pacientes con datos clínicos: {len(pacientes_clinicos_agregados)}")
print(f"Pacientes con imagen: {len(pacientes_con_imagen)}")
print(f"Excluidos por tener SOLO clínico (sin imagen): {len(solo_clinico_excluidos)}")
print(f"Excluidos por tener SOLO imagen (sin clínico, no aparecen en df_paciente por no tener fila clínica): {len(solo_imagen_excluidos)}")
print(f"Intersección final (con AMBOS): {len(interseccion)}")


[FILTRO COHORTE FINAL — verificación explícita]
Pacientes con datos clínicos: 100
Pacientes con imagen: 96
Excluidos por tener SOLO clínico (sin imagen): 8
Excluidos por tener SOLO imagen (sin clínico, no aparecen en df_paciente por no tener fila clínica): 4
Intersección final (con AMBOS): 92


In [29]:
n_antes_filtro = len(df_paciente)
df_paciente['Patient ID'] = df_paciente['Patient ID'].astype(str).str.strip()
df_paciente = df_paciente[df_paciente['Patient ID'].isin(interseccion)].reset_index(drop=True)
n_despues_filtro = len(df_paciente)

In [30]:
print(f"\nPacientes antes del filtro: {n_antes_filtro}")
print(f"Pacientes después del filtro (cohorte con clínico + imagen): {n_despues_filtro}")


Pacientes antes del filtro: 100
Pacientes después del filtro (cohorte con clínico + imagen): 92


In [31]:
## Sección 6. Exclusión de pacientes con metástasis a distancia (M1)

--- Exclusión de pacientes M1 (metástasis a distancia) — CONFIGURABLE ---
El objetivo del estudio se define explícitamente sobre cáncer de mama  LOCALMENTE AVANZADO (estadio III, M0) — M1 corresponde a estadio IV
(metastásico), una población clínicamente distinta (el tratamiento tiene  intención paliativa/control sistémico, no curativa con cirugía posterior).

In [32]:
EXCLUIR_M1 = True

n_antes_M1 = len(df_paciente)
pacientes_M1 = df_paciente[df_paciente['M'] == 1]['Patient ID'].tolist()

if EXCLUIR_M1:
    df_paciente = df_paciente[df_paciente['M'] != 1].reset_index(drop=True)
    print(f"\n[EXCLUSIÓN M1 — criterio 'localmente avanzado', ACTIVADA]")
else:
    print(f"\n[EXCLUSIÓN M1 — DESACTIVADA -- análisis de sensibilidad, M1 incluidos]")

n_despues_M1 = len(df_paciente)
print(f"Pacientes con M=1 (metástasis a distancia) detectados: {len(pacientes_M1)}")
#print(f"IDs con M=1: {pacientes_M1}")
print(f"Cohorte final ({'tras exclusión' if EXCLUIR_M1 else 'M1 incluidos'}): {n_despues_M1}")



[EXCLUSIÓN M1 — criterio 'localmente avanzado', ACTIVADA]
Pacientes con M=1 (metástasis a distancia) detectados: 7
Cohorte final (tras exclusión): 85


In [33]:
# Sufijo para diferenciar los archivos de salida según la variante ejecutada
SUFIJO_VARIANTE = "" if EXCLUIR_M1 else "_conM1"

In [34]:
## Sección 7. Control de calidad y descripción de la cohorte final

In [35]:

# --- Chequeo de seguridad: pacientes sin target válido tras la agregación ---
print("\n[CONTROL DE CALIDAD]")
print("Pacientes con ypN = NA tras agregación:", df_paciente['ypN'].isna().sum())
print("Pacientes con ypT = NA tras agregación:", df_paciente['ypT'].isna().sum())
if df_paciente['ypN'].isna().sum() > 0:
    print("[AVISO] Estos pacientes no tienen ypN en NINGUNA de sus filas originales,")
    print("        no se pueden usar para entrenar el modelo con este target:")
    print(df_paciente[df_paciente['ypN'].isna()]['Patient ID'].tolist())


[CONTROL DE CALIDAD]
Pacientes con ypN = NA tras agregación: 0
Pacientes con ypT = NA tras agregación: 0


In [36]:
print("\nValores únicos de M en la cohorte final (debe ser solo [0.] tras la exclusión):", df_paciente['M'].unique())


Valores únicos de M en la cohorte final (debe ser solo [0.] tras la exclusión): [0.]


In [37]:
print("\nDistribución de ypN (target principal):")
print(df_paciente['ypN'].value_counts(normalize=True))


Distribución de ypN (target principal):
ypN
ypN0      0.647059
ypN2      0.152941
ypN1      0.129412
ypN1mi    0.070588
Name: proportion, dtype: float64


In [38]:
print("\nResumen de categorías terapéuticas recibidas (nº de pacientes por categoría):")
for categoria in CATEGORIAS_FARMACOS:
    col = f'{categoria}'
    if col in df_paciente.columns:
        print(f"  {col}: {df_paciente[col].sum()} pacientes")


Resumen de categorías terapéuticas recibidas (nº de pacientes por categoría):
  taxano: 81 pacientes
  antraciclina: 76 pacientes
  alquilante: 76 pacientes
  platino: 21 pacientes
  anti_HER2: 28 pacientes
  hormonoterapia: 2 pacientes
  inhibidor_CDK46: 1 pacientes
  inmunoterapia: 2 pacientes
  ipatasertib: 0 pacientes
  soporte_GCSF: 9 pacientes


In [39]:
N_FINAL = len(df_paciente)
print(f"\n[RECOMENDACIÓN de predictores por variabilidad, N={N_FINAL}]")
for categoria in CATEGORIAS_FARMACOS:
    col = f'{categoria}'
    if col in df_paciente.columns:
        n_pos = df_paciente[col].sum()
        pct = 100 * n_pos / N_FINAL
        if n_pos == 0 or n_pos == N_FINAL:
            recomendacion = "EXCLUIR (constante, sin variabilidad)"
        elif n_pos < 10:
            recomendacion = f"EXCLUIR (menos de 10 casos, insuficiente para N={N_FINAL})"
        elif pct > 90:
            recomendacion = "Revisar (poca variabilidad, casi todos iguales)"
        else:
            recomendacion = "Mantener como predictor"
        print(f"  {col}: {n_pos} pacientes ({pct:.1f}%) -> {recomendacion}")


[RECOMENDACIÓN de predictores por variabilidad, N=85]
  taxano: 81 pacientes (95.3%) -> Revisar (poca variabilidad, casi todos iguales)
  antraciclina: 76 pacientes (89.4%) -> Mantener como predictor
  alquilante: 76 pacientes (89.4%) -> Mantener como predictor
  platino: 21 pacientes (24.7%) -> Mantener como predictor
  anti_HER2: 28 pacientes (32.9%) -> Mantener como predictor
  hormonoterapia: 2 pacientes (2.4%) -> EXCLUIR (menos de 10 casos, insuficiente para N=85)
  inhibidor_CDK46: 1 pacientes (1.2%) -> EXCLUIR (menos de 10 casos, insuficiente para N=85)
  inmunoterapia: 2 pacientes (2.4%) -> EXCLUIR (menos de 10 casos, insuficiente para N=85)
  ipatasertib: 0 pacientes (0.0%) -> EXCLUIR (constante, sin variabilidad)
  soporte_GCSF: 9 pacientes (10.6%) -> EXCLUIR (menos de 10 casos, insuficiente para N=85)


In [40]:
df_paciente.to_csv(f'data/tabla_clinica_agregada{SUFIJO_VARIANTE}.csv', index=False)
print("\nGuardado en data/tabla_clinica_agregada.csv")
print("Shape final:", df_paciente.shape)


Guardado en data/tabla_clinica_agregada.csv
Shape final: (85, 43)


Reglas orientativas: con N=85 (unos 30 ypN+), una variable binaria con menos de 10 casos
 positivos no permite estimar su efecto de forma estable; con más del 90 % de positivos
 casi no discrimina. Son avisos de apoyo, la decisión final se toma en el modelado.

## Conclusiones

La tabla clínica original tenía 389 filas para 100 pacientes (entre 1 y 6 filas por paciente, una por línea de tratamiento). Se comprobó que ypN y el resto de variables "por paciente" no presentan valores contradictorios entre filas, por lo que tomar el primer valor no vacío es seguro. Los tratamientos se resumieron en 10 familias de fármacos, (9 esquemas oncológicos más factor de soporte GCSF) con una variable binaria y los ciclos de cada una.

La cohorte se redujo de 100 a 92 pacientes al exigir imagen PET-CT (8 sin imagen), y de 92 a **85** al excluir 7 pacientes con metástasis a distancia (M1), para centrar el estudio en cáncer de mama localmente avanzado. No quedan valores perdidos en ypN ni en ypT. El 64,7 % de los pacientes es ypN0 y el 35,3 % tiene afectación ganglionar (ypN1mi, ypN1 o ypN2), unos 30 pacientes: un tamaño de muestra pequeño que condiciona todo el modelado posterior. Cuatro familias de fármacos tienen menos de 10 casos (hormonoterapia, inhibidores de CDK4/6, inmunoterapia y GCSF) y ipatasertib ninguno, por lo que no son utilizables como predictores. El resultado es una tabla de 85 pacientes y 43 columnas.